### Importing libraries


In [50]:
import pandas as pd
import numpy as np
import tensorflow as tf

## Preprocessing data

In [51]:
dataset = pd.read_csv("adult.csv")
print(dataset.head())

   age          workclass  fnlwgt   education  education-num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital-status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital-gain  capital-loss  hours-per-week  native-country  income  
0          2174             0              40   United-States   <=50

In [52]:
X = dataset.iloc[:,:-1]
y = dataset.iloc[:,-1]

In [53]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [54]:
le = LabelEncoder()
y = le.fit_transform(y)
print(y)

[0 0 0 ... 0 0 1]


In [55]:
categorical_columns = ['workclass','education','marital-status','occupation','relationship','race','sex','native-country']

ct = ColumnTransformer(transformers = [('cat',OneHotEncoder(drop='first'),categorical_columns)],remainder = "passthrough")
X = ct.fit_transform(X)
X = X.toarray()
print(X)

[[0.0000e+00 0.0000e+00 0.0000e+00 ... 2.1740e+03 0.0000e+00 4.0000e+01]
 [0.0000e+00 0.0000e+00 0.0000e+00 ... 0.0000e+00 0.0000e+00 1.3000e+01]
 [0.0000e+00 0.0000e+00 0.0000e+00 ... 0.0000e+00 0.0000e+00 4.0000e+01]
 ...
 [0.0000e+00 0.0000e+00 0.0000e+00 ... 0.0000e+00 0.0000e+00 4.0000e+01]
 [0.0000e+00 0.0000e+00 0.0000e+00 ... 0.0000e+00 0.0000e+00 2.0000e+01]
 [0.0000e+00 0.0000e+00 0.0000e+00 ... 1.5024e+04 0.0000e+00 4.0000e+01]]


In [56]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [57]:
numerical_columns = [0,2,4,10,11,12]

scaler = StandardScaler()
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

## Creating ANN

In [58]:
ann = tf.keras.models.Sequential()

In [59]:
ann.add(tf.keras.layers.Dense(units = 6,activation = 'relu'))

In [60]:
ann.add(tf.keras.layers.Dense(units = 6,activation = 'relu'))

In [61]:
ann.add(tf.keras.layers.Dense(units=1,activation='sigmoid'))

### Compiling, Training the ANN Model

In [62]:
ann.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [63]:
print(y_train[:5])

[1 1 0 0 0]


In [64]:
ann.fit(X_train,y_train,batch_size = 32, epochs = 100)

Epoch 1/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6306 - loss: 272.9224
Epoch 2/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6795 - loss: 22.6127
Epoch 3/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6750 - loss: 23.1936
Epoch 4/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6821 - loss: 26.4294
Epoch 5/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6862 - loss: 17.5519
Epoch 6/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6770 - loss: 17.6245
Epoch 7/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6855 - loss: 17.9063
Epoch 8/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6945 - loss: 16.7048
Epoch 9/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6935 - loss: 14.1152
Epoch 10/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6960 - loss: 16.6109
Epoch 11/100
814/814 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6876 - loss: 20.8603
Epoch 12/100
814/814 ━━━━━━━━

In [66]:
from sklearn.metrics import confusion_matrix,classification_report

y_pred = (ann.predict(X_test) > 0.5)

print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 946us/step
[[4695  247]
 [ 729  842]]
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      4942
           1       0.77      0.54      0.63      1571

    accuracy                           0.85      6513
   macro avg       0.82      0.74      0.77      6513
weighted avg       0.84      0.85      0.84      6513

